## NEOs close-approaches clustering

Este proyecto tiene como objetivo aplicar técnicas modernas de machine learning, inteligencia artificial y algoritmos de clustering para analizar datos de Objetos Cercanos a la Tierra (NEOs). A través de estos enfoques, se busca identificar patrones comunes en sus características y realizar una segmentación basada en similitudes dentro del conjunto de datos.

# Data
Los datos fueron extraídos mediante el consumo de la API proporcionada por el sistema de monitoreo de aproximaciones cercanas a la Tierra (Close-Approach Data) del Center for Near Earth Object Studies, perteneciente a la NASA. La fuente oficial se encuentra disponible en: https://cneos.jpl.nasa.gov/ca/

## Librerias data

In [2]:
import pandas as pd
import os
import time
from datetime import datetime
import re
import requests
import math

## Carga y preparacion de datos

In [3]:
# Ruta colab
carpeta_destino = "."
os.makedirs(carpeta_destino, exist_ok=True)
ruta_csv = os.path.join(carpeta_destino, "close_approaches.csv")

In [4]:
# Funciones

def archivo_es_reciente(ruta, dias_maximos=30):
    """Devuelve True si el archivo fue modificado hace menos de dias_maximos días."""
    if not os.path.exists(ruta):
        return False
    edad_segundos = time.time() - os.path.getmtime(ruta)
    return edad_segundos < dias_maximos * 86400


def limpiar_fecha(fecha_str):
    """Elimina sufijos de incertidumbre como ±00:01 en las fechas de la API."""
    if isinstance(fecha_str, str):
        return fecha_str.split("±")[0].strip()
    return fecha_str

In [5]:
# Carga de datos
necesita_descarga = True
df = None

if os.path.exists(ruta_csv):
    print(f"✔ Archivo CSV encontrado en {ruta_csv}")
    if archivo_es_reciente(ruta_csv, dias_maximos=30):
        print("✔ Archivo actualizado (menos de 30 días). Cargando desde archivo local...")
        try:
            df = pd.read_csv(ruta_csv)
            print(f"✔ Datos cargados correctamente. {len(df):,} registros.")
            df['Close-Approach (CA) Date'] = df['Close-Approach (CA) Date'].apply(limpiar_fecha)
            necesita_descarga = False
        except Exception as e:
            print(f"✘ Error al cargar el archivo: {e}")
            print("  Se procederá a descargar datos frescos...")
            necesita_descarga = True
    else:
        print("✘ Archivo desactualizado (más de 30 días). Se descargará de nuevo.")
else:
    print(f"✘ Archivo CSV no encontrado en {ruta_csv}")

 # DESCARGA DESDE LA API DE JPL

if necesita_descarga:
    print("\n⬇ Descargando datos desde la API de JPL (puede tardar unos segundos)...")

    fecha_hoy = datetime.utcnow().strftime('%Y-%m-%d')

    url = (
        f"https://ssd-api.jpl.nasa.gov/cad.api"
        f"?date-min=1900-01-01"
        f"&date-max={fecha_hoy}"
        f"&diameter=true"
    )

    try:
        respuesta = requests.get(url, timeout=60)
        respuesta.raise_for_status()
        datos_json = respuesta.json()

        columnas = datos_json.get('fields', [])
        datos    = datos_json.get('data',   [])
        df = pd.DataFrame(data=datos, columns=columnas)
        print(f"✔ Descarga completada. {len(df):,} registros recibidos.")
        print(f"  Columnas disponibles: {list(df.columns)}")

        # Selección de columnas (solo las que existan en la respuesta)
        columnas_relevantes = ["des", "cd", "dist", "dist_min",
                               "v_rel", "v_inf", "h", "diameter", "diameter_sigma"]
        columnas_relevantes = [c for c in columnas_relevantes if c in df.columns]
        df = df[columnas_relevantes].copy()

        # Conversión numérica
        for col in ["dist", "dist_min", "v_rel", "v_inf", "h", "diameter", "diameter_sigma"]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

        # Diámetro estimado donde falta
        albedo = 1329 / math.sqrt(0.14)
        mascara_sin_diametro = df["diameter"].isna()
        df.loc[mascara_sin_diametro, "diameter"] = (
            albedo * (10 ** (-0.2 * df.loc[mascara_sin_diametro, "h"]))
        )
        df.loc[df["diameter_sigma"].isna(), "diameter_sigma"] = df["diameter"] * 0.35

        # Renombrar y guardar
        df = df.rename(columns={
            "des":            "Object",
            "cd":             "Close-Approach (CA) Date",
            "dist":           "CA DistanceNominal (au)",
            "dist_min":       "CA DistanceMinimum (au)",
            "v_rel":          "V relative(km/s)",
            "v_inf":          "V infinity(km/s)",
            "h":              "H(mag)",
            "diameter":       "Diameter(km)",
            "diameter_sigma": "Std Diameter(km)",
        })

        df.to_csv(ruta_csv, index=False)
        print(f"✔ CSV guardado en {ruta_csv}")

        df['Close-Approach (CA) Date'] = df['Close-Approach (CA) Date'].apply(limpiar_fecha)

    except requests.exceptions.RequestException as e:
        print(f"✘ Error de conexión: {e}")
        raise
    except Exception as e:
        print(f"✘ Error al procesar los datos: {e}")
        raise

if df is None:
    raise Exception("No se pudieron cargar los datos. Verifica la conexión o el archivo.")

print("\n           RESUMEN DE DATOS CARGADOS           ")
print(f"  Total de registros          : {len(df):,}")
print(f"  Nulos en H(mag)             : {df['H(mag)'].isna().sum():,}")
print(f"  Nulos en Diameter(km)       : {df['Diameter(km)'].isna().sum():,}")

print("\n            PRIMEROS 5 REGISTROS     ")
print(df[['Object', 'Diameter(km)', 'CA DistanceNominal (au)']].head().to_string(index=False))

if not necesita_descarga:
    print(f"\n Fuente: archivo local  →  {ruta_csv}")
    print(f"   Última modificación: {time.ctime(os.path.getmtime(ruta_csv))}")
else:
    print(f"\n Fuente: API de JPL (descarga fresca)")

✘ Archivo CSV no encontrado en .\close_approaches.csv

⬇ Descargando datos desde la API de JPL (puede tardar unos segundos)...


C:\Users\Usuario\AppData\Local\Temp\ipykernel_20764\3253108451.py:28: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  fecha_hoy = datetime.utcnow().strftime('%Y-%m-%d')


✔ Descarga completada. 32,345 registros recibidos.
  Columnas disponibles: ['des', 'orbit_id', 'jd', 'cd', 'dist', 'dist_min', 'dist_max', 'v_rel', 'v_inf', 't_sigma_f', 'h', 'diameter', 'diameter_sigma']
✔ CSV guardado en .\close_approaches.csv

           RESUMEN DE DATOS CARGADOS           
  Total de registros          : 32,345
  Nulos en H(mag)             : 8
  Nulos en Diameter(km)       : 7

            PRIMEROS 5 REGISTROS     
    Object  Diameter(km)  CA DistanceNominal (au)
    509352      0.331483                 0.009632
2014 SC324      0.047039                 0.039964
2012 UK171      0.045547                 0.049706
  2024 BA5      0.022206                 0.026434
  2024 BW1      0.033609                 0.037979

 Fuente: API de JPL (descarga fresca)


---

## Datos listos

El dataset procesado ha sido guardado como `close_approaches.csv` en esta misma carpeta (`data/`).

Para continuar con el analisis estadistico y machine learning, ejecutar el notebook `ProyectoNeoRework_ml.ipynb`.